# Lab 01 — NLP Fundamentals

This lab covers the five topics from the first NLP session:

| # | Topic |
| --- | --- |
| 1 | Regular Expressions |
| 2 | Edit Distances |
| 3 | Text Normalization |
| 4 | Feature Extraction |
| 5 | Text Classification |

Work through each section in order. Read the instructions carefully before writing any code.

> **Tip:** Run the **Setup** cell at the top of each section before attempting the exercise.

---
## Exercise 1 — Regular Expressions

**Skills covered:** `re.findall`, `re.sub`, character classes, quantifiers, word boundaries.

In [15]:
# Setup
import re

### Your task

Given the sentence below, do the following:

1. Find all **hashtags** (e.g. `#Python3`).
2. Find all **numbers** (any sequence of digits).
3. Replace every **phone number** (exactly 11 digits) with the string `'PHONE NUM'`,  
   and replace every other number with the string `'NUM'`.
4. Print the three results.

> **Hint:** Handle the 11-digit phone number *before* you replace shorter numbers,  
> or use a substitution helper function that distinguishes them by length.

In [16]:
sentence = "In 2026, I studied #Python3 and #NLP for 2 months , and I loved it! , this is my phone number call me anytime 01147094321 , "

# ============================================================
# WRITE YOUR CODE HERE
# ============================================================

# 1. Find all hashtags
hashtags = re.findall(r"#\w+", sentence)
print("Hashtags:", hashtags)

# 2. Find all numbers
numbers = re.findall(r"\b\d+\b", sentence)
print("Numbers:", numbers)

# 3. Replace phone numbers then other numbers
cleaned = re.sub(r"\d{11}", "PHONE NUM", sentence)
cleaned = re.sub(r"\b\d+\b", "NUM", sentence)
print("Cleaned:", cleaned)

Hashtags: ['#Python3', '#NLP']
Numbers: ['2026', '2', '01147094321']
Cleaned: In NUM, I studied #Python3 and #NLP for NUM months , and I loved it! , this is my phone number call me anytime NUM , 


**Expected output (example):**
```
Hashtags: ['#Python3', '#NLP']
Numbers: ['2026', '2', '01147094321']
Cleaned: In NUM, I studied #Python3 and #NLP for NUM months , and I loved it! , this is my phone number call me anytime PHONE NUM ,
```

---
## Exercise 2 — Edit Distances

**Skills covered:** Levenshtein distance, spelling correction with `min()`.

In [17]:
# Setup — install jellyfish if it is missing
%pip install jellyfish
import jellyfish

### Your task

The user typed `'houze'`. Use the `jellyfish` library to suggest the closest correct spelling.

1. Calculate the **Levenshtein distance** from `typed_word` to every word in `candidates`.
2. Print each candidate and its distance.
3. Use `min()` to find and print the candidate with the **smallest distance**.

Expected suggestion: `house`

In [18]:
typed_word = "houze"
candidates = ["house", "horse", "mouse"]

# ============================================================
# WRITE YOUR CODE HERE
# ============================================================

# 1 & 2. Print each candidate and its Levenshtein distance
suggestions = {}
for candidate in candidates:
    distance = jellyfish.levenshtein_distance(typed_word, candidate)
    suggestions[candidate] = distance
    print(f"{candidate} -> {typed_word} = {distance}")

# 3. Print the closest suggestion
closest = min(suggestions, key=suggestions.get)
print("Suggestion:", closest)

house -> houze = 1
horse -> houze = 2
mouse -> houze = 2
Suggestion: house


**Expected output:**
```
house -> 1
horse -> 2
mouse -> 2
Suggestion: house
```

---
## Exercise 3 — Text Normalization

**Skills covered:** `normalize_text()`, tokenization, stemming with `PorterStemmer`.

In [19]:
# Setup — install nltk if it is missing
# %pip install nltk
import nltk
nltk.download('punkt_tab', quiet=True)
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

In [20]:
def normalize_text(text):
    """Lowercase, remove punctuation, and collapse whitespace."""
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)  # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()  # fix extra spaces
    return text

stemmer = PorterStemmer()

### Your task

1. Use `normalize_text()` to clean the sentence below.  
   Expected result: `'students are studying nlp'`
2. Split the cleaned sentence into a list of words.
3. Apply `stemmer.stem()` to every word.
4. Print the cleaned sentence and the list of stems.

In [21]:
sentence = "  Students ARE studying NLP!!!  "

# ============================================================
# WRITE YOUR CODE HERE
# ============================================================

# 1. Normalize
cleaned = normalize_text(sentence)
print("Cleaned:", cleaned)

# 2. Tokenize
words = word_tokenize(cleaned)
print("Words:", words)

# 3 & 4. Stem each word and print
stems = [stemmer.stem(word) for word in words]
print("Stems:", stems)

Cleaned: students are studying nlp
Words: ['students', 'are', 'studying', 'nlp']
Stems: ['student', 'are', 'studi', 'nlp']


**Expected output:**
```
Cleaned: students are studying nlp
Stems: ['student', 'are', 'studi', 'nlp']
```

---
## Exercise 4 — Feature Extraction (TF-IDF + Cosine Similarity)

**Skills covered:** `TfidfVectorizer`, `cosine_similarity`, document retrieval.

In [22]:
# Setup — install scikit-learn if it is missing
# %pip install scikit-learn
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

### Your task

Find the document that is most similar to the query.

1. Create a `TfidfVectorizer` with English stop words.
2. Fit it on `documents`.
3. Transform **both** the documents and the query.
4. Calculate the cosine similarity between the query vector and each document vector.
5. Print the document with the **highest similarity score**.

Expected document: `'A kitten is a young cat'`

In [23]:
documents = [
    "A kitten is a young cat",
    "Python is used for data science",
    "The car needs more fuel",
]
query = "young kitten and cat"

# ============================================================
# WRITE YOUR CODE HERE
# ============================================================

# 1 & 2. Create and fit the vectorizer
vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit(documents)

# 3. Transform documents and query
doc_vectors  = vectorizer.transform(documents)
query_vector = vectorizer.transform([query])

# 4. Compute cosine similarity
scores = cosine_similarity(query_vector, doc_vectors).flatten()

# 5. Print the best match
best_index = np.argmax(scores)
print("Most similar document:", documents[best_index])

Most similar document: A kitten is a young cat


**Expected output:**
```
Most similar document: A kitten is a young cat
```

---
## Exercise 5 — Text Classification (Sentiment Analysis)

**Skills covered:** `CountVectorizer`, `TfidfVectorizer`, `LogisticRegression`, pipelines, DataFrames.

> **Before you start:** The setup cell below will load the IMDB dataset and train both models.  
> Make sure the dataset file `NLP/datasets/IMDB-Dataset.csv` is available.

In [24]:
# Setup — loads data and trains both models
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

data = pd.read_csv("https://raw.githubusercontent.com/Muhammed-M/iti-summer-training-aug2026/main/NLP/datasets/IMDB-Dataset.csv")
data = data[["review", "sentiment"]].dropna().copy()
data["review"] = data["review"].str.replace("<br />", " ", regex=False)

train_data, test_data = train_test_split(
    data, test_size=0.20, random_state=42, stratify=data["sentiment"]
)
train_data = train_data.reset_index(drop=True)
test_data  = test_data.reset_index(drop=True)

count_model = make_pipeline(
    CountVectorizer(),
    LogisticRegression(max_iter=1000, random_state=42),
)
count_model.fit(train_data["review"], train_data["sentiment"])

tfidf_model = make_pipeline(
    TfidfVectorizer(),
    LogisticRegression(max_iter=1000, random_state=42),
)
tfidf_model.fit(train_data["review"], train_data["sentiment"])

print("Models ready!")

Models ready!


### Your task

Create a small set of movie reviews and compare the two trained models.

1. Write **six short movie reviews** — three positive and three negative.  
   Store them in a list named `student_reviews`.
2. Predict the sentiment of each review with **both** `count_model` and `tfidf_model`.
3. Display the text and both predictions in a **DataFrame** named `student_results`.
4. Answer the reflection questions in the Markdown cell below.

In [26]:
# 1. Write your six movie reviews
student_reviews = [
    # Three positive reviews
    "Flawed script, mediocre film-making, but a decent thriller nonetheles",
    "I never expected it would be that beautiful.",
    "One of the Only Two Films I Watched Twice in Theaters!",
    # Three negative reviews
    "Too slow, it feels like a tv series more than a movie.",
    "Could have been excellent, but bows to conventions",
    "Promising Theme... Disappointing movie"
]

# ============================================================
# WRITE YOUR CODE HERE
# ============================================================

# 2. Predict with both models
count_preds = count_model.predict(student_reviews)
tfidf_preds = tfidf_model.predict(student_reviews)

# 3. Build and display the results DataFrame
student_results = pd.DataFrame({
    "review":           student_reviews,
    "count_prediction": count_preds,
    "tfidf_prediction": tfidf_preds,
})

student_results

,review,count_prediction,tfidf_prediction
0,"Flawed script, mediocre film-making, but a dec...",negative,negative
1,I never expected it would be that beautiful.,positive,positive
2,One of the Only Two Films I Watched Twice in T...,negative,negative
3,"Too slow, it feels like a tv series more than ...",negative,negative
4,"Could have been excellent, but bows to convent...",positive,positive
5,Promising Theme... Disappointing movie,negative,negative


### Reflection questions

Answer directly in this cell after running your code.

1. **Did the two models disagree on any review?** If yes, which one?

   * No, the two models agreed on all reviews so far

2. **Which model gave better predictions for your examples?**

   * I think both models rather performed poorly, with roughly 3/6 reviews being wrongly predicted

3. **Choose one wrong prediction. Why do you think the model made that mistake?**

   *  "Could have been excellent, but bows to conventions" is inherently a negative review, whoever it was predicted as "positive" by both models.

   * A logical explanation might be that Vectorizers in general tend to only assess the literal meaning of words, but completely ignore the inter-words relationships and overall fail to capture the context.

   * That's why in said example, the two models looked at the obvious positive word "Excellent" and diceded it's a positive review, without studying the semantic relationship "Could have been excellent" nor understanding the context "but bows to conventions".